In [1]:
import pandas as pd
import json
import os
from tqdm import tqdm
from glob import glob

In [2]:
y_true = []
with open('../data/splitted_data/fold_1/test.json','r') as textfile:
    for i in json.load(textfile):
        y_true.append(i['ner_tags'])


In [14]:
class ner_eval():
    def __init__(self,true_list,pred_list,eval_dict,inv = False):
        self.true_list = true_list
        self.pred_list = pred_list
        self.eval_dict = eval_dict
        self.inv = inv
        self.true_length = len(true_list)
        self.pred_length = len(pred_list)
        if self.true_length != self.pred_length:
            self.check = False
        else:
            self.check = True        
        self.set_info()
    
    def get_info(self):
        s_idx_list = []
        e_idx_list = []
        ent_list = []
        if self.inv:
            true_list = self.pred_list
        else:
            true_list = self.true_list
        if self.check:
            cnt = -1
            for i in range(len(true_list)):
                if true_list[i] != 'O':
                    label = true_list[i].split('-')
                    if label[0] == 'B':
                        cnt += 1
                        start_idx = i
                        end_idx = i+1
                        s_idx_list.append(start_idx)
                        e_idx_list.append(end_idx)
                        ent = label[1]
                        ent_list.append(ent)
                    elif label[0] == 'I':
                        try:
                            tmp = true_list[i-1]
                            if tmp == 'O':
                                ent_list.append(label[1])
                                s_idx_list.append(i)
                                e_idx_list.append(i+1)
                                cnt += 1
                            elif tmp.split('-')[1] != label[1]:
                                ent_list.append(label[1])
                                s_idx_list.append(i)
                                e_idx_list.append(i+1)
                                cnt += 1
                            elif tmp.split('-')[1] == label[1]:
                                e_idx_list[cnt] = i+1
                        except:
                                ent_list.append(label[1])
                                s_idx_list.append(i)
                                e_idx_list.append(i+1)
                                cnt += 1
                            
            return s_idx_list, e_idx_list, ent_list
                        
    def set_info(self):
        if self.check:
            s_idx_list, e_idx_list, ent_list = self.get_info()
            self.s_idx_list = s_idx_list
            self.e_idx_list = e_idx_list
            self.ent_list = ent_list
            
        
    def strict(self,true,predict,s_idx,e_idx,entity):
        if true[s_idx] != f'B-{entity}' or predict[s_idx] != f'B-{entity}':
            return False
        for idx in range(s_idx,e_idx):
            if true[idx] != predict[idx]:
                return False
        
        if e_idx < len(true) and (true[e_idx] == f'I-{entity}' or predict[e_idx] == f'I-{entity}'):
                return False
        return True
    
    def relax(self,true,predict,s_idx,e_idx,entity):
        for idx in range(s_idx,e_idx):
            try:
                true_ent = true[idx].split('-')[1]
                pred_ent = predict[idx].split('-')[1]
                if true_ent == pred_ent == entity:
                    return True
            except:
                continue
        return False        
    
    def get_strict(self,s_idx,e_idx,ent):
        if self.inv:
            return self.strict(self.pred_list,self.true_list,s_idx,e_idx,ent)
        else:
            return self.strict(self.true_list,self.pred_list,s_idx,e_idx,ent)
    
    def get_relax(self,s_idx,e_idx,ent):
        if self.inv:
            return self.relax(self.pred_list,self.true_list,s_idx,e_idx,ent)
        else:
            return self.relax(self.true_list,self.pred_list,s_idx,e_idx,ent)
        
    def sentence_eval(self):
        result = {}
        cnt = 0
        for i in range(len(self.s_idx_list)):
            s_idx = self.s_idx_list[i]
            e_idx = self.e_idx_list[i]
            ent = self.ent_list[i]
            strict = self.get_strict(s_idx,e_idx,ent)
            relax = self.get_relax(s_idx,e_idx,ent)
            if ent in result.keys():
                result[ent]['strict'] += 1 if strict else 0
                result[ent]['relax'] += 1 if relax else 0
                result[ent]['total'] += 1
            else:
                result[ent] = {}
                result[ent]['strict'] = 1 if strict else 0
                result[ent]['relax'] = 1 if relax else 0
                result[ent]['total'] = 1        
        return result
   
    def update_dict(self):
        if self.check:
            eval_dict = self.eval_dict
            result = self.sentence_eval()
            for k,v in result.items():
                if k in eval_dict.keys():
                    eval_dict[k]['strict'] += v['strict']
                    eval_dict[k]['relax'] += v['relax']
                    eval_dict[k]['total'] += v['total']
                else:
                    eval_dict[k] = v
            return eval_dict
        else:
            return self.eval_dict
        
    def sum_and_fulfilldict(self):
        eval_dict = self.eval_dict
        targeted_category = ['Adherence','Alcohol','Concern','Drug','Education','Employment','Financial','Healthcare','Insurance','Literacy','Living','MentalHealth','Recommendation','Smoke','Social','SubstanceUse','Transportation','Trauma']
        missing_category = list(set(targeted_category) - set(eval_dict.keys()))
        for i in missing_category:
            eval_dict[i] = {}
            eval_dict[i]['strict'] = 0
            eval_dict[i]['relax'] = 0
            eval_dict[i]['total'] = 0
        strict = 0
        relax = 0
        total = 0
        for k,v in eval_dict.items():
            strict += v['strict']
            relax += v['relax']
            total += v['total']
        tmp = {}
        tmp['strict'] = strict
        tmp['relax'] = relax
        tmp['total'] = total
        eval_dict['overall'] = tmp
        return dict(sorted(eval_dict.items()))


In [6]:
def eval_metric(true_eval,pred_eval):
    metrics = {}
    for i in range(len(true_eval)):
        k = list(true_eval.keys())[i]
        if true_eval[k]['total'] == 0:
            recall_s = -1
        else:
            recall_s = true_eval[k]['relax']/true_eval[k]['total']
            
        if pred_eval[k]['total'] == 0:
            precision_s = -1
        else:
            precision_s = true_eval[k]['relax']/pred_eval[k]['total']

#         TP_s = true_eval[k]['strict']
#         TN_s = true_eval[k]['total'] - TP_s
#         FP_s = pred_eval[k]['total'] - pred_eval[k]['strict']
#         FN_s = TP_s - pred_eval[k]['strict']
#         precision_s = TP_s / (TP_s + FP_s)
#         recall_s = TP_s / (TP_s + FN_s) 
#         accuracy_s = (TP_s + TN_s)/ (TP_s + TN_s + FP_s + FN_s)
        if (precision_s + recall_s) == 0:
            f1_s = 'N/A'
        else:
            f1_s = 2 * precision_s * recall_s / (precision_s + recall_s)
        
        if true_eval[k]['total'] == 0:
            recall_r = -1
        else:
            recall_r = true_eval[k]['relax']/true_eval[k]['total']
        if pred_eval[k]['total'] == 0:
            precision_r = -1
        else:
            precision_r = true_eval[k]['relax']/pred_eval[k]['total']
#         TP_r = true_eval[k]['relax']
#         TN_r = true_eval[k]['total'] - TP_r
#         FP_r = pred_eval[k]['total'] - pred_eval[k]['relax']
#         FN_r = TP_r - pred_eval[k]['relax']
#         precision_r = TP_r / (TP_r + FP_r)
#         accuracy_r = (TP_r + TN_r)/ (TP_r + TN_r + FP_r + FN_r)
#         recall_r = TP_r / (TP_r + FN_r) 
        if (precision_r + recall_r) == 0:
            f1_r = 'N/A'
        else:
            f1_r = 2 * precision_r * recall_r / (precision_r + recall_r)

        
        metrics[k] = {'strict':{'precision':precision_s,'recall':recall_s,'f-1':f1_s},
                      'relax':{'precision':precision_r,'recall':recall_r,'f-1':f1_r,}}
        
    return metrics

In [7]:
def fix_bio_tags(bio_lists):
    fixed = []
    for tag_list in bio_lists:
        new_tags = []
        for i, tag in enumerate(tag_list):
            # If tag is 'O', nothing to change.
            if tag == 'O':
                new_tags.append(tag)
            else:
                prefix, entity = tag.split('-', 1)
                # If first token, or previous token is 'O', or previous token is a different entity,
                # then this token should be a beginning (B-) tag.
                if i == 0 or tag_list[i-1] == 'O' or (tag_list[i-1] != 'O' and tag_list[i-1].split('-', 1)[1] != entity):
                    new_tags.append('B-' + entity)
                else:
                    # Otherwise, continue the entity span as an inside (I-) tag.
                    new_tags.append('I-' + entity)
        fixed.append(new_tags)
    return fixed

def refine_bio_tags(bio_lists):
    refined = []
    for tags in bio_lists:
        new_tags = tags[:]  # work on a copy
        # Iterate from the second token to the second-to-last token.
        for i in range(1, len(new_tags) - 1):
            # Look for an "O" token.
            if new_tags[i] == 'O':
                prev_tag = new_tags[i - 1]
                next_tag = new_tags[i + 1]
                # Check if the previous token is part of an entity and the next token starts an entity.
                if prev_tag != 'O' and next_tag.startswith('B-'):
                    prev_entity = prev_tag.split('-', 1)[1]
                    next_entity = next_tag.split('-', 1)[1]
                    # If both tokens refer to the same entity, fill in the gap and adjust the following token.
                    if prev_entity == next_entity:
                        new_tags[i] = 'I-' + prev_entity
                        new_tags[i + 1] = 'I-' + next_entity
        refined.append(new_tags)
    return refined


In [8]:
def get_sentence_metric(y_pred, y_true):
    pred_sdoh = []
    true_sdoh = []
    for i, j in zip(y_pred, y_true):
        i = [x.split('-')[-1] for x in i]
        j = [x.split('-')[-1] for x in j]
        pred_sdoh.append(list(set(i) - set(['O'])))
        true_sdoh.append(list(set(j) - set(['O'])))
    tp, fp, fn = 0, 0, 0
    for p,t in zip(pred_sdoh, true_sdoh):
        p = set(p)
        t = set(t)
        tp += len(p & t) 
        fp += len(p - t)
        fn += len(t - p)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * (precision * recall) / (precision + recall)
    return precision, recall ,f1

In [22]:
def evaluate_sentence_level(y_true, y_pred, categories, current_dict):
    from collections import defaultdict
    
    # Initialize counters
    results = {cat: {'TP': 0, 'FP': 0, 'FN': 0} for cat in categories}

    # Iterate sentence by sentence
    for true_tags, pred_tags in zip(y_true, y_pred):
        true_entities = set()
        pred_entities = set()

        # Extract unique categories present in y_true
        for tag in true_tags:
            if tag.startswith('B-') or tag.startswith('I-'):
                category = tag.split('-')[-1]
                if category in categories:
                    true_entities.add(category)
                    
        # Extract unique categories present in y_pred
        for tag in pred_tags:
            if tag.startswith('B-') or tag.startswith('I-'):
                category = tag.split('-')[-1]
                if category in categories:
                    pred_entities.add(category)

        # Check for TP, FP, FN
        for cat in categories:
            if cat in true_entities and cat in pred_entities:
                results[cat]['TP'] += 1
            if cat not in true_entities and cat in pred_entities:
                results[cat]['FP'] += 1
            if cat in true_entities and cat not in pred_entities:
                results[cat]['FN'] += 1
    
    # Calculate precision, recall, F1 per category

    for cat in categories:
        TP = results[cat]['TP']
        FP = results[cat]['FP']
        FN = results[cat]['FN']
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        current_dict[cat]['sentence-level'] = {'precision': round(precision,3), 'recall': round(recall,3), 'f-1': round(f1,3)}
    
    return current_dict


In [21]:
targeted_category = ['Adherence','Alcohol','Concern','Drug','Education','Employment','Financial','Healthcare','Insurance','Literacy','Living','MentalHealth','Recommendation','Smoke','Social','SubstanceUse','Transportation','Trauma']

In [24]:
for fold in range(1,6):
    for d in glob(f'../output/*fold_{fold}*/predictions/'):
        y_true = []
        with open(f'../data/splitted_data/fold_{fold}/test.json','r') as textfile:
            for i in json.load(textfile):
                y_true.append(i['ner_tags'])
        overall_eval = {}
        predict_file = d + '/predictions.txt'
        output_file = d + '/evaluation.json'
        y_predict = []
        with open(predict_file) as txtfile:
            for i in txtfile.readlines():
                y_predict.append(i.split())
        y_predict = refine_bio_tags(fix_bio_tags(y_predict))        
        eval_dict = {}
        eval_dict2 = {}

        for i in range(len(y_true)):
            try:
                eval_dict = ner_eval(y_true[i],y_predict[i],eval_dict).update_dict()
            except:
                print(i)
                continue
        eval_dict = ner_eval(y_true[i],y_predict[i],eval_dict).sum_and_fulfilldict()

        for i in range(len(y_true)):
            try:
                eval_dict_2 = ner_eval(y_true[i],y_predict[i],eval_dict2,True).update_dict()
            except:
                print(i)
                continue
        eval_dict2 = ner_eval(y_true[i],y_predict[i],eval_dict2,True).sum_and_fulfilldict()
        overall_eval[d] = eval_metric(eval_dict,eval_dict2)
        overall_eval[d]['sentence'] = {}
        overall_eval[d]['sentence']['precision'], overall_eval[d]['sentence']['recall'],overall_eval[d]['sentence']['f-1'] = get_sentence_metric(y_predict, y_true)
        overall_eval[d] = evaluate_sentence_level(y_true, y_predict, targeted_category, overall_eval[d])
        
        # with open(output_file,'w') as f:
        #     json.dump(overall_eval[d],f,indent=3)
        # for k,v in eval_metric(eval_dict,eval_dict2).items():
        #     print(k)
        #     print(f"strict: {v['strict']}")
        #     print(f"relax: {v['relax']}")
        


In [10]:
perf_dict = {}
fold_num = 5
for model in ['BioBERT', 'roberta', 'BERT']:
    tmp_dict = {}
    tmp_dict['strict'] = {}
    tmp_dict['relax'] = {}
    tmp_dict['sentence'] = {}
    tmp_dict['strict']['precision'], tmp_dict['strict']['recall'], tmp_dict['strict']['f-1'] = 0, 0, 0
    tmp_dict['relax']['precision'], tmp_dict['relax']['recall'], tmp_dict['relax']['f-1'] = 0, 0, 0
    tmp_dict['sentence']['precision'], tmp_dict['sentence']['recall'], tmp_dict['sentence']['f-1'] = 0, 0, 0
    for path in glob(f'../output/{model}*/predictions/evaluation.json'):
        val = json.load(open(path,'r'))
        
        tmp_dict['strict']['precision'] += val['overall']['strict']['precision'] / fold_num
        tmp_dict['strict']['recall'] += val['overall']['strict']['recall'] / fold_num
        tmp_dict['strict']['f-1'] += val['overall']['strict']['f-1'] / fold_num
        
        tmp_dict['relax']['precision'] += val['overall']['relax']['precision'] / fold_num
        tmp_dict['relax']['recall'] += val['overall']['relax']['recall'] / fold_num
        tmp_dict['relax']['f-1'] += val['overall']['relax']['f-1'] / fold_num
        
        tmp_dict['sentence']['precision'] += val['sentence']['precision'] / fold_num
        tmp_dict['sentence']['recall'] += val['sentence']['recall'] / fold_num
        tmp_dict['sentence']['f-1'] += val['sentence']['f-1'] / fold_num
    perf_dict[model] = tmp_dict
with open(f'../output/model_performance.json', 'w') as f:
    json.dump(perf_dict, f)